In [14]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1,2,3'
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [15]:
yaml = '''task: trivial_mcq
dataset_path: arbml/ArabicMMLU
output_type: multiple_choice
test_split: test
doc_to_text: "Question: {{Question}}.\nChoices: A.{{A}} B.{{B}} C.{{C}} D.{{D}} E.{{E}}.\nAnswer:"
doc_to_target: answer
doc_to_choice: ['A', 'B', 'C', 'D', 'E']
metric_list:
  - metric: acc
    aggregation: mean
    higher_is_better: True
  - metric: acc_norm
    aggregation: mean
    higher_is_better: true
metadata:
  version: 1.0'''


In [16]:
!mkdir -p eval_harness_extra_tasks/trivial_mcq
def save_yaml(yaml_text):
  with open('./eval_harness_extra_tasks/trivial_mcq/trivial_mcq.yaml', 'w') as f:
    f.write(yaml_text)

save_yaml(yaml)

In [17]:
from lm_eval.models.huggingface import HFLM
from transformers import AutoModelForCausalLM, AutoTokenizer

In [18]:
MODEL_PATH = "/hdd/shared_models/AceGPT-7B"
TOKENIZER_PATH = MODEL_PATH

In [19]:
if 'model' not in locals():
    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, trust_remote_code=True, device_map="auto")
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

In [20]:
lm_obj = HFLM(
    pretrained=model,
    trust_remote_code=True,
    # parallelize=True,
    device_map="auto",
    tokenizer=tokenizer,
    batch_size=8,
)

2024-11-14:18:40:39,831 WARNING  [huggingface.py:96] `pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
2024-11-14:18:40:39,832 INFO     [huggingface.py:483] Using model type 'default'
2024-11-14:18:40:39,850 WARNING  [huggingface.py:277] Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration


In [21]:
import lm_eval

In [22]:
def evaluate_tasks():
    # MAKE SURE THE NOTEBOOK IS RUNNING FROM THE PROJECT ROOT!
    task_manager = lm_eval.tasks.TaskManager(include_path="./eval_harness_extra_tasks/trivial_mcq")
    results = lm_eval.simple_evaluate(  # call simple_evaluate
        model=lm_obj,
        tasks=['trivial_mcq'],
        num_fewshot=0,
        task_manager=task_manager,
    )
    return results

In [23]:
results = evaluate_tasks()

2024-11-14:18:40:46,602 INFO     [evaluator.py:164] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2024-11-14:18:40:46,602 INFO     [evaluator.py:217] Using pre-initialized model
2024-11-14:18:40:51,116 WARNING  [task.py:325] [Task: trivial_mcq] has_training_docs and has_validation_docs are False, using test_docs as fewshot_docs but this is not recommended.
2024-11-14:18:40:51,118 WARNING  [task.py:325] [Task: trivial_mcq] has_training_docs and has_validation_docs are False, using test_docs as fewshot_docs but this is not recommended.
2024-11-14:18:40:52,228 WARNING  [evaluator.py:270] Overwriting default num_fewshot of trivial_mcq from None to 0
2024-11-14:18:40:52,230 INFO     [task.py:415] Building contexts for trivial_mcq on rank 0...
100%|██████████| 14575/14575 [00:09<00:00, 1527.40it/s]
2024-11-14:18:41:02,886 INFO     [evaluator.py:489] Running loglikelihood requests
Running loglikelihood requests:

In [24]:
print(lm_eval.utils.make_table(results))

|   Tasks   |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|-----------|------:|------|-----:|--------|---|-----:|---|-----:|
|trivial_mcq|      1|none  |     0|acc     |↑  |0.2815|±  |0.0037|
|           |       |none  |     0|acc_norm|↑  |0.2815|±  |0.0037|



In [25]:
results

{'results': {'trivial_mcq': {'acc,none': 0.28150943396226413,
   'acc_stderr,none': 0.003725354699588584,
   'acc_norm,none': 0.28150943396226413,
   'acc_norm_stderr,none': 0.003725354699588584}},
 'group_subtasks': {'trivial_mcq': []},
 'configs': {'trivial_mcq': {'task': 'trivial_mcq',
   'dataset_path': 'arbml/ArabicMMLU',
   'test_split': 'test',
   'doc_to_text': 'Question: {{Question}}. Choices: A.{{A}} B.{{B}} C.{{C}} D.{{D}} E.{{E}}. Answer:',
   'doc_to_target': 'answer',
   'doc_to_choice': ['A', 'B', 'C', 'D', 'E'],
   'description': '',
   'target_delimiter': ' ',
   'fewshot_delimiter': '\n\n',
   'num_fewshot': 0,
   'metric_list': [{'metric': 'acc',
     'aggregation': 'mean',
     'higher_is_better': True},
    {'metric': 'acc_norm', 'aggregation': 'mean', 'higher_is_better': True}],
   'output_type': 'multiple_choice',
   'repeats': 1,
   'should_decontaminate': False,
   'metadata': {'version': 1.0}}},
 'versions': {'trivial_mcq': 1.0},
 'n-shot': {'trivial_mcq': 0},

In [26]:
# exit()